In [5]:
%load_ext autoreload
%autoreload 2 

# AISDI - ćwiczenie 2
## Wprowadzenie
Celem drugiego zadania z przedmiotu Algorytmy i Struktury Danych (AISDI) była implementacja oraz analiza wydajności algorytmów grafowych na przykładzie miejskiej sieci drogowej. Głównym zadaniem było zaimportowanie danych zawierających odległości oraz czasy przejazdu między węzłami w różnych porach dnia, a następnie wyznaczenie optymalnych tras na podstawie kryterium dystansu oraz czasu.

## Zaimportowane biblioteki
Zgodnie z wymaganiami projektowymi, do reprezentacji grafu nie wykorzystano gotowych bibliotek (takich jak networkx). Graf został zaimplementowany od podstaw przy użyciu zmodyfikowanej listy sąsiedztwa.

Główną strukturą przechowującą dane jest słownik, którego kluczem jest identyfikator węzła (miasta), a wartością lista obiektów klasy Destination.
Zastosowane narzędzia i moduły:

- csv – do wczytywania i parsowania plików z danymi.

- destination.py – autorska klasa reprezentująca krawędź grafu (cel podróży) wraz z jej wagami (dystans, czas w zależności od pory dnia).

- graph_logic.py – moduł zawierający implementację algorytmów trasowania.

- time – do przeprowadzania pomiarów wydajności i czasu kompilacji algorytmów.

In [ ]:
import csv
from pprint import pprint
from destination import Destination
from graph_logic import *
import time

## Analiza danych wejściowych
Dane wejściowe pochodzą z pliku city_small.csv. Każdy wiersz reprezentuje krawędź grafu nieskierowanego (dwukierunkową drogę) wraz z zestawem wag. Struktura danych prezentuje się następująco:

In [7]:
with open("data/city_small.csv", newline="") as csv_file:
    small_cities = csv.reader(csv_file)
    for index, row in enumerate(small_cities):
        if index > 5:
            break
        print(row)

['od', 'do', 'odleglosc', 'czas_rano', 'czas_poludnie', 'czas_wieczor']
['0', '9', '17.1', '19', '11', '14']
['0', '20', '6.84', '11', '5', '8']
['1', '11', '9.4', '22', '12', '12']
['1', '21', '8.12', '13', '11', '15']
['1', '22', '19.55', '33', '20', '20']


Konstrukcja grafu odbywa się poprzez iterację po pliku CSV i dwukrotne dodanie krawędzi (od A do B oraz od B do A) do słownika pathways za pomocą metody create_destination().


## Wyznaczanie najkrótszej ścieżki

Tak wygląda proces importowania miast z pliku, następnie tworzymy polączenia miedzy nimi wykorzystujac metode create_destination(), ktoej implementacja znajduje sie w pliku graph_logic.py

In [29]:
pathways = {}
with open("data/city_small.csv", newline="") as csv_file:
    small_cities = csv.reader(csv_file)
    header = next(small_cities)
    
    for index, path in enumerate(small_cities):
        create_destination(pathways, path[0], path[1], float(path[2]), int(path[3]), int(path[4]), int(path[5]))
        create_destination(pathways, path[1], path[0], float(path[2]), int(path[3]), int(path[4]), int(path[5]))


Następnie zbadamy czas znajdywania najkrótszej drogi od poszczególnych punktów, wraz z ich czasem kompilacji.


In [30]:
start_time = time.perf_counter()
cities_info = get_dijkstra_info_distance(49, pathways)
end_time = time.perf_counter()
total_time = end_time - start_time

print_dijkstra_info(cities_info)
print()
print(f"Total time for small city dijkstra is {total_time:.04f}")

Distance from 49 to 10 is 23.99
Distance from 9 to 29 is inf
Distance from 7 to 34 is 57.16
Distance from 45 to 17 is 27.96
Distance from 49 to 0 is inf
Distance from 18 to 4 is 85.14
Distance from 40 to 36 is 42.95
Distance from 7 to 26 is inf
Distance from 10 to 13 is 110.89
Distance from 8 to 13 is 52.19

Total time for small city dijkstra is 0.0005


Na tym etapie porównamy średni czas podróży 
Algorytm wykonuje się stosunkowo szybko.
### Czas
- zaprezentowano algortym osobnej dijstry dla czasu
- liczy od w trakcie godzine/czas obecny dla grafu
- w trakcie iteracji po grafie zmienia godzine i od niej pobiera odpowiedni czas

In [31]:
start_time = time.perf_counter()
cities_info = get_dijkstra_info_time(49, pathways, "07:30")
end_time = time.perf_counter()
total_time = end_time - start_time

print_dijkstra_info(cities_info)
print()
print(f"Total time for small city dijkstra time is {total_time:.04f}")
print(f"This simulation is for starting time equal 07:30")

Arrival time from 49 to 18 is 08:26
Arrival time from 48 to 42 is 09:46
Arrival time from 12 to 31 is 08:53
Arrival time from 41 to 24 is inf
Arrival time from 15 to 6 is 10:12
Arrival time from 38 to 41 is inf
Arrival time from 27 to 25 is 09:05
Arrival time from 28 to 25 is 09:09
Arrival time from 30 to 38 is 07:59
Arrival time from 20 to 14 is inf

Total time for small city dijkstra time is 0.0008
This simulation is for starting time equal 07:30


### Komiwojazer:
Ten etap był najtrudniejszy do zrealizowania, po pierwsze w wielu przypadkach dotyczących grafu nie mam możliwości przejscia z jednegoe miasta do drugiego bezposrednio, dlatego trzeba zawracać. Dodatkowo niektóre miasta  tworzą graf, który jest odizolowany od głownego grafu, co powoduje, że klaysyczne rozwiązanie problemu kowimojażera staje się niemożliwe, a do tego należy wspomnieć, że tym przypadku została użyta heurystyka, bo dla zwykłego komputeraz obliczenie poprawnej najkrótszej drogi zajmuje n! czasu, dla kilku miast komputer szybko znajdzie najlepszą trasę, ale dla kilkudziesięciu miast potrzebny czas wynosi więcej niż czas istnienie wszechświata. W tym przypadku została użyta najprostrza heurystyka, która nie jest optymalna.

Tak prezentują się wyniki heurystyki, w której to algortym szuka najkrótszej drogi dla grafu małego miasta:

In [33]:
start_time = time.perf_counter()
start_city = "21"
road, distance = distance_of_travelling_salesman(pathways, start_city)
end_time = time.perf_counter()
total_time = end_time - start_time
print(f"Cities that were visited from {start_city} are: ")
print(road)
print(f"and the distance is {distance}")
print(f"It took programme {total_time} for solving travelling salesman problem")

Cities that were visited from 21 are: 
['21', '39', '1', '34', '25', '42', '11', '4', '33', '44', '13', '6', '22', '23', '47', '29', '5', '8', '45', '28', '17', '40', '24', '37', '46', '38', '43', '48', '30', '19', '14', '18', '10', '15', '32', '2', '27', '7', '31', '49', '12', '36', '21']
and the distance is 631.46
It took programme 0.049390750005841255 for solving travelling salesman problem


## Średnie Miasto